## Autonomous Traders Dashboard

A central hub to manage dedicated Crypto, Stocks, and Forex agents leveraging the MCP tool ecosystem.

In [1]:
import sys
import os

# Append the parent directory '6_mcp' to the system path to seamlessly reuse local modules
parent_path = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if parent_path not in sys.path:
    sys.path.append(parent_path)

# Change the CWD so that sqlite DB outputs and MCP server calls map to the correct local environment
if os.path.basename(os.getcwd()) != "6_mcp":
    os.chdir(parent_path)

In [5]:
from accounts import Account
from database import read_log
from util import css, js, Color
from traders import Trader
import plotly.express as px
import pandas as pd
import asyncio
import gradio as gr
from market import is_market_open
from tracers import LogTracer
from agents import add_trace_processor
from dotenv import load_dotenv

load_dotenv(override=True)

ModuleNotFoundError: No module named 'accounts'

In [ ]:
names = ["Crypto", "Stocks", "Forex"]
lastnames = ["Alpha", "Beta", "Gamma"]
model_names = ["gpt-4o-mini", "gpt-4o-mini", "gpt-4o-mini"]
short_model_names = ["GPT-4o Mini", "GPT-4o Mini", "GPT-4o Mini"]

crypto_strategy = """
You are Crypto Alpha, an elite autonomous trader solely focusing on cryptocurrency assets (BTC, ETH, SOL, etc).
You leverage momentum and technical indicators across markets to identify explosive growth opportunities.
Your goal is to maximize PnL through active crypto swing trading, rebalancing quickly when momentum shifts.
"""

stocks_strategy = """
You are Stocks Beta, a seasoned equity trader specializing in US technological and industrial stocks (AAPL, MSFT, TSLA, etc).
You are a value-oriented investor attempting to find mispriced equities with long-term holds and dividend growth.
Your goal is to sustain a robust and risk-adjusted portfolio.
"""

forex_strategy = """
You are Forex Gamma, a macro trader focusing on foreign exchange parity and global indicators.
You track central bank rates, geopolitical shifts, and currency valuations (EUR, USD, GBP, JPY).
You prioritize stable, low-spread FX pairs exploiting arbitrage and macro-economic announcements.
NOTE: Use currency ETFs (e.g., UUP, FXE, FXY) to take Forex positions.
"""

def reset_traders():
    Account.get("Crypto").reset(crypto_strategy)
    Account.get("Stocks").reset(stocks_strategy)
    Account.get("Forex").reset(forex_strategy)

reset_traders()

In [ ]:
# Create Trader Instances
traders = [Trader(name, lastname, model_name) for name, lastname, model_name in zip(names, lastnames, model_names)]

# Async Engine configuration
RUN_EVERY_N_MINUTES = 60
RUN_MARKET_CLOSED = True # Set to True for testing purposes out-of-hours
is_running = False

async def engine_loop():
    global is_running
    add_trace_processor(LogTracer())
    while True:
        if is_running:
            if RUN_MARKET_CLOSED or is_market_open():
                print("Running trader cycle...")
                await asyncio.gather(*[trader.run() for trader in traders])
            else:
                print("Market is closed, skipping run")
        await asyncio.sleep(RUN_EVERY_N_MINUTES * 60)

# Start engine thread
loop = asyncio.get_event_loop()
loop.create_task(engine_loop())

def toggle_trading():
    global is_running
    is_running = not is_running
    return f"Trading Active: {is_running}"

In [ ]:
mapper = {
    "trace": Color.WHITE,
    "agent": Color.CYAN,
    "function": Color.GREEN,
    "generation": Color.YELLOW,
    "response": Color.MAGENTA,
    "account": Color.RED,
}

class DashboardTrader:
    def __init__(self, name: str, lastname: str, model_name: str):
        self.name = name
        self.lastname = lastname
        self.model_name = model_name
        self.account = Account.get(name)

    def reload(self):
        self.account = Account.get(self.name)

    def get_title(self) -> str:
        return f"<div style='text-align: center;font-size:34px;'>{self.name} <span style='color:#ccc;font-size:24px;'>({self.model_name}) - {self.lastname}</span></div>"

    def get_portfolio_value(self):
        portfolio_value = self.account.calculate_portfolio_value() or 0.0
        pnl = self.account.calculate_profit_loss(portfolio_value) or 0.0
        return portfolio_value, pnl

    def get_portfolio_html(self) -> str:
        portfolio_value, pnl = self.get_portfolio_value()
        color = "green" if pnl >= 0 else "red"
        emoji = "⬆" if pnl >= 0 else "⬇"
        return f"<div style='text-align: center;'><span style='font-size:32px; color: {color};'>${portfolio_value:,.0f}</span><br/><span style='font-size:24px; color: {color};'>{emoji} ${pnl:,.0f} PNL</span></div>"

    def get_portfolio_value_chart(self):
        df = pd.DataFrame(self.account.portfolio_value_time_series, columns=["datetime", "value"])
        if df.empty:
            df = pd.DataFrame([{"datetime": pd.Timestamp.now(), "value": 10000.0}])
        df["datetime"] = pd.to_datetime(df["datetime"])
        fig = px.line(df, x="datetime", y="value")
        fig.update_layout(height=250, margin=dict(l=40, r=20, t=20, b=40), paper_bgcolor="#f0f0f0", plot_bgcolor="#fff")
        fig.update_xaxes(tickformat="%m/%d", tickangle=45, tickfont=dict(size=8))
        fig.update_yaxes(tickfont=dict(size=8), tickformat=",.0f")
        return fig

    def get_holdings_df(self) -> pd.DataFrame:
        holdings = self.account.get_holdings()
        if not holdings:
            return pd.DataFrame(columns=["Symbol", "Quantity"])
        return pd.DataFrame([{"Symbol": k, "Quantity": v} for k, v in holdings.items()])

    def get_transactions_df(self) -> pd.DataFrame:
        transactions = self.account.list_transactions()
        if not transactions:
            return pd.DataFrame(columns=["Timestamp", "Symbol", "Quantity", "Price", "Rationale"])
        return pd.DataFrame(transactions)

    def get_logs(self, previous=None) -> str:
        logs = read_log(self.name, last_n=13)
        response = ""
        for log in logs:
            timestamp, log_type, message = log
            color = mapper.get(log_type, Color.WHITE).value
            response += f"<span style='color:{color}'>{timestamp} : [{log_type}] {message}</span><br/>"
        response = f"<div style='height:250px; overflow-y:auto; background: #111; padding: 10px; border-radius: 8px;'>{response}</div>"
        return response if response != previous else gr.update()

In [ ]:
def create_dashboard_ui():
    agents = [DashboardTrader(n, l, m) for n, l, m in zip(names, lastnames, short_model_names)]

    with gr.Blocks(title="Autonomous Traders Hub", css=css, js=js, theme=gr.themes.Soft(primary_hue="emerald")) as ui:
        gr.Markdown("# 📈 Global Autonomous Trading Hub")

        # Central Dashboard Metrics
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### System Controls")
                engine_status = gr.Textbox(label="Engine Status", value="Trading Active: False", interactive=False)
                toggle_btn = gr.Button("Toggle Trading Engine", variant="primary")
                toggle_btn.click(fn=toggle_trading, outputs=engine_status)
            
            with gr.Column(scale=2):
                gr.Markdown("### Total Aggregated Metrics")
                with gr.Row():
                    total_equity_html = gr.HTML()
                    total_pnl_html = gr.HTML()

        gr.Markdown("---")
        gr.Markdown("### Trading Agents Dashboard")
        
        portfolio_htmls = []
        charts = []
        holdings = []
        transactions = []
        logs = []

        with gr.Tabs():
            for agent in agents:
                with gr.TabItem(f"{agent.name} ({agent.lastname})"):
                    gr.HTML(agent.get_title())
                    
                    with gr.Row():
                        with gr.Column(scale=1):
                            p_html = gr.HTML(agent.get_portfolio_html)
                            portfolio_htmls.append(p_html)
                        with gr.Column(scale=3):
                            chart = gr.Plot(agent.get_portfolio_value_chart, container=True, show_label=False)
                            charts.append(chart)
                    
                    with gr.Row(variant="panel"):
                        log = gr.HTML(agent.get_logs)
                        logs.append(log)
                    
                    with gr.Row():
                        holding_table = gr.Dataframe(value=agent.get_holdings_df, label="Holdings", headers=["Symbol", "Quantity"], row_count=(5, "dynamic"))
                        holdings.append(holding_table)
                        
                    with gr.Row():
                        tx_table = gr.Dataframe(value=agent.get_transactions_df, label="Transactions", headers=["Timestamp", "Symbol", "Quantity", "Price", "Rationale"], row_count=(5, "dynamic"))
                        transactions.append(tx_table)

        def refresh_all():
            total_eq = 0
            total_pnl = 0
            outs = []
            
            for agent in agents:
                agent.reload()
                val, pnl = agent.get_portfolio_value()
                total_eq += val
                total_pnl += pnl
                
            global_color = "green" if total_pnl >= 0 else "red"
            global_emoji = "⬆" if total_pnl >= 0 else "⬇"
            eq_html = f"<div style='text-align: center; border: 2px solid {global_color}; padding: 10px; border-radius: 8px;'><h2>Total Equity</h2><span style='font-size:32px; color: {global_color};'>${total_eq:,.0f}</span></div>"
            pnl_html = f"<div style='text-align: center; border: 2px solid {global_color}; padding: 10px; border-radius: 8px;'><h2>Total PNL</h2><span style='font-size:32px; color: {global_color};'>{global_emoji} ${total_pnl:,.0f}</span></div>"
            
            outs.extend([eq_html, pnl_html])
            for agent in agents:
                outs.append(agent.get_portfolio_html())
                outs.append(agent.get_portfolio_value_chart())
                outs.append(agent.get_holdings_df())
                outs.append(agent.get_transactions_df())
                
            return tuple(outs)

        def refresh_logs():
            return tuple([agent.get_logs() for agent in agents])

        refresh_timer = gr.Timer(value=60)
        refresh_timer.tick(
            fn=refresh_all,
            inputs=[],
            outputs=[total_equity_html, total_pnl_html] + portfolio_htmls + charts + holdings + transactions,
            show_progress="hidden"
        )
        
        log_timer = gr.Timer(value=1.5)
        log_timer.tick(
            fn=refresh_logs,
            inputs=[],
            outputs=logs,
            show_progress="hidden"
        )
        
        ui.load(refresh_all, [], [total_equity_html, total_pnl_html] + portfolio_htmls + charts + holdings + transactions)

    return ui

ui = create_dashboard_ui()
ui.launch(inbrowser=True)